# SG-SC / CGOS Assignment 2 — Final Cross-Dataset Evaluation

**Status of this file: NOT YET EXECUTED.** It contains no results. Run it top-to-bottom on a Colab GPU; it writes every result file itself. Nothing in it is copied from the team's sample table.

**What is evaluated**

* Models: Original GPT-2, then **Remove**, **Average** and **NonlinearMultiply** for 1–5 simultaneously merged layer pairs.
* **NonlinearMultiply = my finalized Assignment 2 model = CGOS** (Calibration-Guided Operator Selection). For each pair it picks Remove / Average / Multiply by lowest hidden-state MSE on the WikiText-2 *calibration* split. No test data touches it. LinearMultiply is not tested (the team lead said to ignore it) and its column stays empty.
* Datasets: DBpedia-14 (test), AG News (test), IMDb (test), **plus** the existing WikiText-2 validation/test as an extra, clearly separated evaluation.

Fitting, affine maps, CGOS selection, `AffineMergedBlock` and the WikiText-2 PPL convention are reused from `SGSC_Assignment2_CGOS_FULL.ipynb` unchanged. The existing notebook is not modified.

In [1]:
!pip install -q -U datasets accelerate "pandas==2.2.3" openpyxl
!pip install -q transformers==5.16.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 65.7 MB/s eta 0:00:00


## 0. Configuration (the only things you may want to edit)

In [2]:
import os, gc, math, random, time, json, sys, platform, subprocess, datetime
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import transformers, datasets as datasets_pkg
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

# ---- identical to the finalized Assignment 2 notebook ----
MODEL_ID = "openai-community/gpt2"
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FIT_TOKENS, CALIB_TOKENS, EVAL_TOKENS = 20000, 20000, 50000   # MODE="full" values
EVAL_BLOCK_SIZE = 512
STATE_CHUNK_SIZE = 256
RIDGE = 1e-4

# ---- new for this evaluation ----
# Team notation -> Python block index. The user asked for 1-based translation
# (team 1-2 -> Python blocks 0,1). Set to "0-based" ONLY if the team lead says so.
TEAM_NUMBERING = "1-based"
BATCH_BLOCKS = 16            # 512-token blocks per forward pass; lower it if you hit CUDA OOM
SEPARATOR_TEXT = "\n\n"      # same document joiner as the notebook's join_nonempty()
SMOKE = os.environ.get("SGSC_SMOKE", "0") == "1"   # SGSC_SMOKE=1 -> 300 docs/dataset, outputs get a _SMOKE suffix
SMOKE_DOCS = 300
OUT_DIR = "/content/final_submission" if os.path.isdir("/content") else "./final_submission"
os.makedirs(OUT_DIR, exist_ok=True)
SUFFIX = "_SMOKE" if SMOKE else ""

TEAM_PLANS = {
    1: [(3, 4)],
    2: [(1, 2), (4, 5)],
    3: [(1, 2), (4, 5), (8, 9)],
    4: [(1, 2), (4, 5), (6, 7), (8, 9)],
    5: [(1, 2), (4, 5), (6, 7), (8, 9), (10, 11)],
}
METHODS = ["Remove", "Average", "NonlinearMultiply"]

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed()
torch.backends.cuda.matmul.allow_tf32 = False   # keep fp32 numerics for every model
torch.backends.cudnn.allow_tf32 = False

print("device:", DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(CPU - this will be extremely slow)")
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets_pkg.__version__)
if SMOKE: print("*** SMOKE MODE: results are NOT final ***")

device: cuda Tesla T4
torch 2.11.0+cu130 | transformers 5.16.1 | datasets 5.0.1


## 1. Existing Assignment 2 components (reused unchanged) + WikiText-2

In [3]:
# ---------- WikiText-2: same loading/joining as the finalized notebook ----------
wikitext = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
def join_nonempty(split):
    return "\n\n".join(x for x in split["text"] if x is not None and x.strip())
train_text = join_nonempty(wikitext["train"])
validation_text = join_nonempty(wikitext["validation"])
test_text = join_nonempty(wikitext["test"])
calib_text = train_text[len(train_text) // 2:]

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.model_max_length = int(1e12)

def load_fresh_model():
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
    model.config.use_cache = False
    model.eval()
    return model

original_model = load_fresh_model().to(DEVICE)
N_BLOCKS_ORIG = len(original_model.transformer.h)

# ---------- existing WikiText-2 PPL convention (verbatim logic) ----------
@torch.no_grad()
def calculate_ppl(model, text, block_size=EVAL_BLOCK_SIZE, max_tokens=EVAL_TOKENS):
    model.eval()
    ids = tokenizer(text, return_tensors="pt", add_special_tokens=False)["input_ids"][0]
    if max_tokens is not None:
        ids = ids[:max_tokens]
    n_blocks = len(ids) // block_size
    if n_blocks == 0:
        raise ValueError("Not enough tokens for one evaluation block.")
    total_nll, total_pred_tokens = 0.0, 0
    for i in range(n_blocks):
        x = ids[i * block_size:(i + 1) * block_size].unsqueeze(0).to(DEVICE)
        outputs = model(input_ids=x, labels=x, use_cache=False)
        count = x.numel() - 1
        total_nll += outputs.loss.item() * count
        total_pred_tokens += count
    return math.exp(total_nll / total_pred_tokens)

def parameter_count(model):
    return sum(p.numel() for p in model.parameters())

def free_model(model):
    del model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def _as_tensor(output):
    return output[0] if isinstance(output, (tuple, list)) else output

@torch.no_grad()
def collect_pair_states(model, text, L, total_tokens=FIT_TOKENS, chunk_size=STATE_CHUNK_SIZE):
    ids = tokenizer(text, return_tensors="pt", add_special_tokens=False)["input_ids"][0]
    ids = ids[:total_tokens]
    X_list, Y1_list, Y2_list = [], [], []
    d = model.config.n_embd
    def pre_hook_L(module, inputs):
        X_list.append(inputs[0].detach().reshape(-1, d).cpu().float())
    def post_hook_L(module, inputs, output):
        Y1_list.append(_as_tensor(output).detach().reshape(-1, d).cpu().float())
    def post_hook_L1(module, inputs, output):
        Y2_list.append(_as_tensor(output).detach().reshape(-1, d).cpu().float())
    handles = [
        model.transformer.h[L].register_forward_pre_hook(pre_hook_L),
        model.transformer.h[L].register_forward_hook(post_hook_L),
        model.transformer.h[L + 1].register_forward_hook(post_hook_L1),
    ]
    try:
        for start in range(0, len(ids), chunk_size):
            x = ids[start:start + chunk_size]
            if x.numel() < 2: continue
            _ = model(x.unsqueeze(0).to(DEVICE), use_cache=False)
    finally:
        for h in handles: h.remove()
    return torch.cat(X_list, 0), torch.cat(Y1_list, 0), torch.cat(Y2_list, 0)

def fit_affine(X, Y, ridge=RIDGE):
    X = X.double(); Y = Y.double()
    x_mean = X.mean(dim=0, keepdim=True); y_mean = Y.mean(dim=0, keepdim=True)
    Xc = X - x_mean; Yc = Y - y_mean
    gram = Xc.T @ Xc
    diag_scale = gram.diag().mean().clamp_min(1e-12)
    reg = ridge * diag_scale * torch.eye(gram.shape[0], dtype=gram.dtype)
    A = torch.linalg.solve(gram + reg, Xc.T @ Yc)
    b = (y_mean - x_mean @ A).squeeze(0)
    return A.float(), b.float()

def hidden_mse(X, Y_true, A, b):
    with torch.no_grad():
        return torch.mean((X @ A + b - Y_true) ** 2).item()

def cgos_select_operator(X_calib, Y2_calib, A1, b1, A2, b2):
    """Calibration-Guided Operator Selection (finalized Assignment 2 method)."""
    cands = {
        "Remove":   (A2, b2),
        "Average":  (0.5 * A1 + 0.5 * A2, 0.5 * b1 + 0.5 * b2),
        "Multiply": (A1 @ A2, b1 @ A2 + b2),
    }
    errors = {k: hidden_mse(X_calib, Y2_calib, *v) for k, v in cands.items()}
    selected = min(errors, key=errors.get)
    A, b = cands[selected]
    return A, b, {"Selected": selected, "E_remove": errors["Remove"], "E_avg": errors["Average"], "E_mul": errors["Multiply"]}

class AffineMergedBlock(nn.Module):
    def __init__(self, A, b):
        super().__init__()
        d = A.shape[0]
        self.linear = nn.Linear(d, d)
        with torch.no_grad():
            self.linear.weight.copy_(A.T)
            self.linear.bias.copy_(b)
    def forward(self, hidden_states, *args, **kwargs):
        return self.linear(hidden_states)

baseline_params = parameter_count(original_model)
print("Original GPT-2 blocks:", N_BLOCKS_ORIG, "| parameters:", f"{baseline_params:,}")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Original GPT-2 blocks: 12 | parameters: 124,439,808


## 2. Layer-number mapping and merge plans

The team document never states its indexing convention. Following your instruction, team notation is treated as **1-based**, so `team a-b` means Python blocks `a-1, b-1`. Team notation is what appears in the final tables; the Python blocks are stored in separate audit columns. Switch `TEAM_NUMBERING` in cell 0 if the team lead says otherwise.

In [5]:
OFFSET = 1 if TEAM_NUMBERING == "1-based" else 0
def team_pair_to_L(a, b):
    assert b == a + 1, f"pair {a}-{b} is not adjacent"
    L = a - OFFSET
    assert 0 <= L <= N_BLOCKS_ORIG - 2, f"team pair {a}-{b} maps outside GPT-2 blocks"
    return L

PLANS = {}
for steps, pairs in TEAM_PLANS.items():
    Ls = [team_pair_to_L(a, b) for a, b in pairs]
    assert Ls == sorted(Ls) and all(Ls[i + 1] >= Ls[i] + 2 for i in range(len(Ls) - 1)), "overlapping/unsorted pairs"
    PLANS[steps] = {
        "team_plan": "; ".join(f"{a}-{b}" for a, b in pairs),
        "python_pairs": Ls,
        "python_plan": "; ".join(f"blocks {L},{L+1}" for L in Ls),
    }
print("TEAM_NUMBERING =", TEAM_NUMBERING)
for s, p in PLANS.items():
    print(f"steps={s}: team [{p['team_plan']}]  ->  Python [{p['python_plan']}]")
UNIQUE_L = sorted({L for p in PLANS.values() for L in p["python_pairs"]})
print("unique Python pair starts to fit:", UNIQUE_L)

TEAM_NUMBERING = 1-based
steps=1: team [3-4]  ->  Python [blocks 2,3]
steps=2: team [1-2; 4-5]  ->  Python [blocks 0,1; blocks 3,4]
steps=3: team [1-2; 4-5; 8-9]  ->  Python [blocks 0,1; blocks 3,4; blocks 7,8]
steps=4: team [1-2; 4-5; 6-7; 8-9]  ->  Python [blocks 0,1; blocks 3,4; blocks 5,6; blocks 7,8]
steps=5: team [1-2; 4-5; 6-7; 8-9; 10-11]  ->  Python [blocks 0,1; blocks 3,4; blocks 5,6; blocks 7,8; blocks 9,10]
unique Python pair starts to fit: [0, 2, 3, 5, 7, 9]


## 3. Fit the per-pair affine maps and run CGOS (WikiText-2 train / calibration only)

Each pair is fitted **once** on the *original* GPT-2's hidden states exactly as in the finalized notebook, then reused wherever that pair appears. This is not tuning on the new datasets: DBpedia, AG News and IMDb are never touched here. Because the maps are fitted on the original model, later pairs in a multi-pair model are **not** refitted after earlier merges (a disclosed limitation).

In [6]:
PAIR_MAPS, CGOS_LOG = {}, []
for L in UNIQUE_L:
    X, Y1, Y2 = collect_pair_states(original_model, train_text, L, total_tokens=FIT_TOKENS)
    A1, b1 = fit_affine(X, Y1)
    A2, b2 = fit_affine(Y1, Y2)
    Xc, Y1c, Y2c = collect_pair_states(original_model, calib_text, L, total_tokens=CALIB_TOKENS)
    A_c, b_c, meta = cgos_select_operator(Xc, Y2c, A1, b1, A2, b2)
    PAIR_MAPS[L] = {
        "Remove": (A2, b2),
        "Average": (0.5 * A1 + 0.5 * A2, 0.5 * b1 + 0.5 * b2),
        "NonlinearMultiply": (A_c, b_c),
        "cgos_selected": meta["Selected"],
    }
    CGOS_LOG.append({"python_pair": f"{L}-{L+1}", "team_pair": f"{L+OFFSET}-{L+1+OFFSET}", **meta})
    del X, Y1, Y2, Xc, Y1c, Y2c
cgos_df = pd.DataFrame(CGOS_LOG)
print(cgos_df.to_string(index=False))

python_pair team_pair Selected  E_remove    E_avg    E_mul
        0-1       1-2 Multiply  8.796502 3.689672 0.416217
        2-3       3-4 Multiply 18.651186 5.999968 0.552724
        3-4       4-5 Multiply  0.952363 0.824334 0.600349
        5-6       6-7 Multiply  1.190043 1.134599 0.991942
        7-8       8-9 Multiply  2.586210 2.515502 2.121503
       9-10     10-11 Multiply  7.568981 9.907519 6.366787


## 4. Build every merged model from a fresh GPT-2

In [4]:
def build_multi_pair_model(pair_starts, method):
    """Fresh GPT-2; every listed pair replaced by ONE AffineMergedBlock; all other blocks untouched."""
    Ls = sorted(pair_starts)
    assert all(Ls[i + 1] >= Ls[i] + 2 for i in range(len(Ls) - 1)), "overlapping pairs"
    model = load_fresh_model()
    blocks = list(model.transformer.h)
    assert len(blocks) == N_BLOCKS_ORIG
    new_blocks, i, Lset = [], 0, set(Ls)
    while i < len(blocks):
        if i in Lset:
            A, b = PAIR_MAPS[i][method]
            new_blocks.append(AffineMergedBlock(A, b)); i += 2
        else:
            new_blocks.append(blocks[i]); i += 1
    assert len(new_blocks) == N_BLOCKS_ORIG - len(Ls)
    model.transformer.h = nn.ModuleList(new_blocks)
    model.config.n_layer = len(new_blocks)
    model.config.use_cache = False
    model.eval()
    return model.to(DEVICE)

## 5. Team datasets (exact loading spec) and the PPL protocol

**Protocol, identical for every model on a given dataset** (the existing WikiText-2 convention, extended from "first 50,000 tokens" to the *whole* test split):

* tokenizer: GPT-2 BPE, `add_special_tokens=False`, no truncation;
* every non-empty document is tokenized separately and the documents are joined with the token(s) of `"\n\n"` (the same joiner as `join_nonempty`), giving one long token stream;
* the stream is cut into **non-overlapping blocks of 512 tokens**, with no stride and no overlap; the final incomplete block is dropped;
* loss = mean next-token NLL over all 511 predictions per block, token-weighted over the whole split; PPL = exp(total NLL / total predicted tokens), fp32;
* dataset labels are never read for PPL.

The only difference from the existing WikiText-2 convention is full split instead of the first 50k tokens (the first 50k tokens would cover under 2% of DBpedia and only a few hundred IMDb reviews). The existing WikiText-2 numbers are still produced with the unchanged `calculate_ppl`.

In [7]:
dbpedia = load_dataset("fancyzhx/dbpedia_14")
test_dbpedia = dbpedia["test"]
test_dbpedia = test_dbpedia.map(lambda x: {"text": x["title"] + " " + x["content"]})

agnews = load_dataset("fancyzhx/ag_news")
test_agnews = agnews["test"]

imdb = load_dataset("stanfordnlp/imdb")
test_imdb = imdb["test"]

DATASETS = {"DBpedia": test_dbpedia, "AG News": test_agnews, "IMDb": test_imdb}
EXPECTED_SIZES = {"DBpedia": 70000, "AG News": 7600, "IMDb": 25000}
for k, v in DATASETS.items():
    print(f"{k:8s} test size = {len(v):,}")
    if not SMOKE: assert len(v) == EXPECTED_SIZES[k], f"{k}: unexpected size {len(v)}"

SEP_IDS = tokenizer(SEPARATOR_TEXT, add_special_tokens=False)["input_ids"]

def build_blocks(ds, name):
    texts = ds["text"]                      # the label column is never accessed here
    if SMOKE: texts = texts[:SMOKE_DOCS]
    pieces, n_docs, first = [], 0, True
    for s in range(0, len(texts), 1000):
        chunk = [t for t in texts[s:s + 1000] if t is not None and t.strip()]
        if not chunk: continue
        for ids in tokenizer(chunk, add_special_tokens=False)["input_ids"]:
            if not first: pieces.append(np.asarray(SEP_IDS, dtype=np.int32))
            pieces.append(np.asarray(ids, dtype=np.int32)); first = False; n_docs += 1
    stream = np.concatenate(pieces)
    n_blocks = len(stream) // EVAL_BLOCK_SIZE
    blocks = stream[:n_blocks * EVAL_BLOCK_SIZE].reshape(n_blocks, EVAL_BLOCK_SIZE)
    info = {"dataset": name, "split_size": len(ds), "docs_used": n_docs, "tokens_total": int(len(stream)),
            "eval_blocks": int(n_blocks), "tokens_evaluated": int(n_blocks * EVAL_BLOCK_SIZE),
            "tokens_dropped_tail": int(len(stream) - n_blocks * EVAL_BLOCK_SIZE),
            "predicted_tokens": int(n_blocks * (EVAL_BLOCK_SIZE - 1))}
    return blocks, info

BLOCKS, DATA_INFO = {}, {}
for name, ds in DATASETS.items():
    BLOCKS[name], DATA_INFO[name] = build_blocks(ds, name)
    print(DATA_INFO[name])

@torch.no_grad()
def ppl_on_blocks(model, blocks, bs=BATCH_BLOCKS):
    model.eval()
    nll, cnt = 0.0, 0
    V = model.config.vocab_size
    for s in range(0, len(blocks), bs):
        x = torch.from_numpy(blocks[s:s + bs]).long().to(DEVICE)
        logits = model(input_ids=x, use_cache=False).logits
        loss = F.cross_entropy(logits[:, :-1].reshape(-1, V).float(), x[:, 1:].reshape(-1), reduction="sum")
        nll += loss.item(); cnt += x.shape[0] * (x.shape[1] - 1)
        del logits, loss, x
    return math.exp(nll / cnt), nll, cnt

README.md:   0%|          | 0.00/7.64k [00:00<?, ?B/s]

dbpedia_14/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  106MB            

dbpedia_14/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

dbpedia_14/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 13.3MB            

dbpedia_14/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DBpedia  test size = 70,000
AG News  test size = 7,600
IMDb     test size = 25,000
{'dataset': 'DBpedia', 'split_size': 70000, 'docs_used': 70000, 'tokens_total': 5112918, 'eval_blocks': 9986, 'tokens_evaluated': 5112832, 'tokens_dropped_tail': 86, 'predicted_tokens': 5102846}
{'dataset': 'AG News', 'split_size': 7600, 'docs_used': 7600, 'tokens_total': 400107, 'eval_blocks': 781, 'tokens_evaluated': 399872, 'tokens_dropped_tail': 235, 'predicted_tokens': 399091}
{'dataset': 'IMDb', 'split_size': 25000, 'docs_used': 25000, 'tokens_total': 7345542, 'eval_blocks': 14346, 'tokens_evaluated': 7345152, 'tokens_dropped_tail': 390, 'predicted_tokens': 7330806}


## 6. Sanity checks (all must pass before the full run)

In [8]:
checks = []
def check(name, ok, detail=""):
    checks.append({"check": name, "passed": bool(ok), "detail": str(detail)})
    print(("PASS " if ok else "FAIL ") + name + (f"  [{detail}]" if detail != "" else ""))
    assert ok, f"sanity check failed: {name}"

check("original GPT-2 parameter count == 124,439,808", baseline_params == 124_439_808, f"{baseline_params:,}")
check("original has 12 blocks", N_BLOCKS_ORIG == 12)

# parameter counts / structure for every plan and method
prev = baseline_params
param_by_steps = {}
for steps in sorted(PLANS):
    Ls = PLANS[steps]["python_pairs"]
    assert len(set(Ls)) == len(Ls)
    m = build_multi_pair_model(Ls, "NonlinearMultiply")
    p = parameter_count(m)
    param_by_steps[steps] = p
    check(f"{steps}-pair model: {N_BLOCKS_ORIG - steps} blocks", len(m.transformer.h) == N_BLOCKS_ORIG - steps)
    check(f"{steps}-pair model has fewer parameters than the previous one", p < prev, f"{p:,}")
    prev = p
    # untouched blocks must be identical to the original model
    merged_idx = {L + o for L in Ls for o in (0, 1)}
    j = 0; ok = True; i = 0
    while i < N_BLOCKS_ORIG:
        if i in {L for L in Ls}:
            ok &= isinstance(m.transformer.h[j], AffineMergedBlock); i += 2
        else:
            for (n1, p1), (n2, p2) in zip(m.transformer.h[j].named_parameters(), original_model.transformer.h[i].named_parameters()):
                ok &= torch.equal(p1.detach().cpu(), p2.detach().cpu())
            i += 1
        j += 1
    check(f"{steps}-pair model: untouched blocks identical to original, merged blocks are affine", ok)
    with torch.no_grad():
        out = m(input_ids=torch.from_numpy(BLOCKS["AG News"][:1]).long().to(DEVICE), use_cache=False).logits
    check(f"{steps}-pair model forward pass finite", torch.isfinite(out).all().item())
    del out; free_model(m)
check("5-pair model has the largest reduction", param_by_steps[5] == min(param_by_steps.values()))

# PPL machinery: batched implementation == existing per-block HF-loss implementation
ref_blocks = BLOCKS["AG News"][:8]
ppl_fast, _, _ = ppl_on_blocks(original_model, ref_blocks)
with torch.no_grad():
    nll = cnt = 0.0
    for b in ref_blocks:
        x = torch.from_numpy(b).long().unsqueeze(0).to(DEVICE)
        nll += original_model(input_ids=x, labels=x, use_cache=False).loss.item() * (x.numel() - 1); cnt += x.numel() - 1
ppl_ref = math.exp(nll / cnt)
check("batched PPL matches per-block HF-loss PPL (rel diff < 1e-4)", abs(ppl_fast - ppl_ref) / ppl_ref < 1e-4, f"{ppl_fast:.6f} vs {ppl_ref:.6f}")
ppl_fast2, _, _ = ppl_on_blocks(original_model, ref_blocks)
check("repeat evaluation is (near-)deterministic", abs(ppl_fast - ppl_fast2) / ppl_fast < 1e-6, f"{ppl_fast:.8f} vs {ppl_fast2:.8f}")
check("PPL finite", math.isfinite(ppl_fast))

# tokenization identical for every model: one tokenizer, blocks built once and shared
check("one shared token-block array per dataset (identical tokenization for all models)", all(len(BLOCKS[k]) > 0 for k in BLOCKS))
check("labels not used: build_blocks reads only the 'text' column", True, "see build_blocks()")

# reproduces the existing finalized-notebook WikiText-2 numbers
orig_val = calculate_ppl(original_model, validation_text)
orig_test = calculate_ppl(original_model, test_text)
print(f"original WikiText-2 validation PPL = {orig_val:.6f} (finalized notebook: ~32.904)")
print(f"original WikiText-2 test PPL       = {orig_test:.6f} (finalized notebook: ~38.486758)")
check("existing WikiText-2 baselines reproduced (|diff| < 0.05)", abs(orig_val - 32.904) < 0.05 and abs(orig_test - 38.486758) < 0.05)

mem0 = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
check("no model leaked after sanity builds (GPU mem < 1.5 GB)", mem0 < 1.5, f"{mem0:.2f} GB")
print("\nAll sanity checks passed.")

PASS original GPT-2 parameter count == 124,439,808  [124,439,808]
PASS original has 12 blocks


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

PASS 1-pair model: 11 blocks
PASS 1-pair model has fewer parameters than the previous one  [110,854,656]
PASS 1-pair model: untouched blocks identical to original, merged blocks are affine
PASS 1-pair model forward pass finite


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

PASS 2-pair model: 10 blocks
PASS 2-pair model has fewer parameters than the previous one  [97,269,504]
PASS 2-pair model: untouched blocks identical to original, merged blocks are affine
PASS 2-pair model forward pass finite


...

PASS 4-pair model: 8 blocks
PASS 4-pair model has fewer parameters than the previous one  [70,099,200]
PASS 4-pair model: untouched blocks identical to original, merged blocks are affine
PASS 4-pair model forward pass finite


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

PASS 5-pair model: 7 blocks
PASS 5-pair model has fewer parameters than the previous one  [56,514,048]
PASS 5-pair model: untouched blocks identical to original, merged blocks are affine
PASS 5-pair model forward pass finite
PASS 5-pair model has the largest reduction


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


PASS batched PPL matches per-block HF-loss PPL (rel diff < 1e-4)  [52.816525 vs 52.816538]
PASS repeat evaluation is (near-)deterministic  [52.81652461 vs 52.81652461]
PASS PPL finite
PASS one shared token-block array per dataset (identical tokenization for all models)
PASS labels not used: build_blocks reads only the 'text' column  [see build_blocks()]
original WikiText-2 validation PPL = 32.904634 (finalized notebook: ~32.904)
original WikiText-2 test PPL       = 38.486758 (finalized notebook: ~38.486758)
PASS existing WikiText-2 baselines reproduced (|diff| < 0.05)
PASS no model leaked after sanity builds (GPU mem < 1.5 GB)  [0.74 GB]

All sanity checks passed.


## 7. Full evaluation: 16 models × (DBpedia, AG News, IMDb) + existing WikiText-2

Progress is saved after every model, so if Colab disconnects you can re-run the notebook and it resumes from `raw_progress.json`. Each merged model is built from a fresh GPT-2 and freed afterwards. Nothing here depends on test-set results.

In [9]:
PROGRESS = os.path.join(OUT_DIR, f"raw_progress{SUFFIX}.json")
done = json.load(open(PROGRESS)) if os.path.exists(PROGRESS) else {}
ALL_EVALS = list(DATASETS) + ["WikiText-2 (validation)", "WikiText-2 (test)"]

registry = [("Original GPT-2", "Original", 0, [])]
for steps in sorted(PLANS):
    for method in METHODS:
        registry.append((f"{method}|{steps}", method, steps, PLANS[steps]["python_pairs"]))

t0 = time.time()
for key, method, steps, Ls in registry:
    if key in done and all(d in done[key]["ppl"] for d in ALL_EVALS):
        print("skip (done):", key); continue
    model = original_model if method == "Original" else build_multi_pair_model(Ls, method)
    rec = {"method": method, "steps": steps, "python_pairs": Ls,
           "params": parameter_count(model), "n_blocks_model": len(model.transformer.h), "ppl": {}, "nll": {}, "pred_tokens": {}}
    for name in DATASETS:
        p, nll, cnt = ppl_on_blocks(model, BLOCKS[name])
        assert math.isfinite(p), f"non-finite PPL: {key} on {name}"
        rec["ppl"][name], rec["nll"][name], rec["pred_tokens"][name] = p, nll, cnt
    rec["ppl"]["WikiText-2 (validation)"] = calculate_ppl(model, validation_text)
    rec["ppl"]["WikiText-2 (test)"] = calculate_ppl(model, test_text)
    done[key] = rec
    json.dump(done, open(PROGRESS, "w"))
    print(f"[{time.time()-t0:7.0f}s] {key:28s} params={rec['params']:,}  " + "  ".join(f"{d}={rec['ppl'][d]:.3f}" for d in ALL_EVALS))
    if method != "Original": free_model(model)
print("Evaluation finished.")

[   1235s] Original GPT-2               params=124,439,808  DBpedia=31.364  AG News=63.345  IMDb=40.305  WikiText-2 (validation)=32.905  WikiText-2 (test)=38.487


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[   2332s] Remove|1                     params=110,854,656  DBpedia=50.653  AG News=134.854  IMDb=76.151  WikiText-2 (validation)=57.169  WikiText-2 (test)=65.861


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[   3430s] Average|1                    params=110,854,656  DBpedia=44.905  AG News=110.061  IMDb=64.598  WikiText-2 (validation)=48.936  WikiText-2 (test)=56.947


...

[  12274s] Remove|5                     params=56,514,048  DBpedia=218439.002  AG News=83262.017  IMDb=17793.263  WikiText-2 (validation)=64309.240  WikiText-2 (test)=45219.233


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[  12806s] Average|5                    params=56,514,048  DBpedia=75484.810  AG News=40525.913  IMDb=10260.918  WikiText-2 (validation)=17045.298  WikiText-2 (test)=14222.673


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[  13348s] NonlinearMultiply|5          params=56,514,048  DBpedia=1929.048  AG News=3124.123  IMDb=1427.375  WikiText-2 (validation)=652.733  WikiText-2 (test)=666.434
Evaluation finished.


## 8. Assemble tables and write all result files

In [10]:
orig = done["Original GPT-2"]
ORIG_PARAMS = orig["params"]
raw_rows = []
for key, rec in done.items():
    for d in ALL_EVALS:
        ppl, oppl = rec["ppl"][d], orig["ppl"][d]
        s = rec["steps"]
        raw_rows.append({
            "dataset": d,
            "method": "Original GPT-2" if rec["method"] == "Original" else rec["method"],
            "steps": s,
            "merge_plan_team_notation": "None" if s == 0 else PLANS[s]["team_plan"],
            "merge_plan_python_blocks": "None" if s == 0 else PLANS[s]["python_plan"],
            "numbering_assumed": TEAM_NUMBERING,
            "ppl": ppl, "original_ppl": oppl,
            "degradation_pct": (ppl - oppl) / oppl * 100.0,
            "parameter_count": rec["params"],
            "parameter_reduction_pct": (ORIG_PARAMS - rec["params"]) / ORIG_PARAMS * 100.0,
            "blocks_in_model": rec["n_blocks_model"],
            "eval_docs": DATA_INFO[d]["docs_used"] if d in DATA_INFO else None,
            "eval_tokens": DATA_INFO[d]["tokens_evaluated"] if d in DATA_INFO else EVAL_TOKENS,
            "predicted_tokens": rec["pred_tokens"].get(d),
            "cgos_selected_operators": ("; ".join(f"{L}-{L+1}:{PAIR_MAPS[L]['cgos_selected']}" for L in rec["python_pairs"]) if rec["method"] == "NonlinearMultiply" else ""),
        })
raw_df = pd.DataFrame(raw_rows)
raw_df["model_label"] = raw_df["method"].replace({"NonlinearMultiply": "NonlinearMultiply (my CGOS model)"})

# parameter reduction must be method-independent for a given step count
pr = raw_df[raw_df.steps > 0].groupby("steps")["parameter_reduction_pct"].nunique()
assert (pr == 1).all(), "parameter reduction differs between methods at the same step count"
assert (raw_df.groupby("steps")["parameter_reduction_pct"].first().diff().dropna() > 0).all(), "reduction must grow with steps"

THREE = ["DBpedia", "AG News", "IMDb"]
# --- Table A: degradation (team format; LinearMultiply left empty) ---
a = raw_df[(raw_df.steps > 0) & raw_df.dataset.isin(THREE)].pivot_table(index=["dataset", "steps"], columns="method", values="degradation_pct").reset_index()
a["dataset"] = pd.Categorical(a["dataset"], THREE, ordered=True)
a = a.sort_values(["dataset", "steps"]).reset_index(drop=True)
table_a = pd.DataFrame({"Dataset": a["dataset"].astype(str), "Steps (number of merged pairs)": a["steps"],
                        "LinearMultiply (not tested)": "", "Remove": a["Remove"], "Average": a["Average"], "NonlinearMultiply": a["NonlinearMultiply"]})
table_a_fmt = table_a.copy()
for c in ["Remove", "Average", "NonlinearMultiply"]:
    table_a_fmt[c] = table_a[c].map(lambda v: f"{v:.1f}%")

# --- Table B: proposed model details (NonlinearMultiply) ---
def ppl_of(method, steps, d):
    return raw_df[(raw_df.method == method) & (raw_df.steps == steps) & (raw_df.dataset == d)]["ppl"].iloc[0]
rows_b = [{"Steps": "Original", "Merge plan": "-", "Parameter reduction": "0.00%",
           **{f"{d} PPL": f"{ppl_of('Original GPT-2', 0, d):.2f}" for d in THREE}}]
for s in sorted(PLANS):
    pr_s = raw_df[(raw_df.method == "NonlinearMultiply") & (raw_df.steps == s)]["parameter_reduction_pct"].iloc[0]
    rows_b.append({"Steps": s, "Merge plan": PLANS[s]["team_plan"], "Parameter reduction": f"{pr_s:.2f}%",
                   **{f"{d} PPL": f"{ppl_of('NonlinearMultiply', s, d):.2f}" for d in THREE}})
table_b = pd.DataFrame(rows_b)

# --- Model table for all methods (raw PPL, auditable) ---
rows_m = []
for key, rec in done.items():
    s = rec["steps"]
    rows_m.append({"Model": "Original GPT-2" if rec["method"] == "Original" else rec["method"], "Steps": s,
                   "Merge plan": "None" if s == 0 else PLANS[s]["team_plan"],
                   "Parameter reduction (%)": (ORIG_PARAMS - rec["params"]) / ORIG_PARAMS * 100.0,
                   **{f"{d} PPL": rec["ppl"][d] for d in THREE}})
model_table = pd.DataFrame(rows_m)

# --- WikiText-2 existing evaluation kept separate from the new datasets ---
wiki_df = raw_df[raw_df.dataset.str.startswith("WikiText-2")][["dataset", "method", "steps", "merge_plan_team_notation", "ppl", "original_ppl", "degradation_pct", "parameter_reduction_pct"]]

# --- metadata ---
def _git():
    try: return subprocess.check_output(["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL).decode().strip()
    except Exception: return None
metadata = {
    "status": "SMOKE TEST - NOT FINAL" if SMOKE else "full evaluation executed",
    "timestamp_utc": datetime.datetime.utcnow().isoformat() + "Z",
    "model_checkpoint": MODEL_ID, "tokenizer": MODEL_ID + " (GPT-2 BPE, add_special_tokens=False)",
    "datasets": {"DBpedia": "fancyzhx/dbpedia_14 test (text = title + ' ' + content)", "AG News": "fancyzhx/ag_news test",
                 "IMDb": "stanfordnlp/imdb test", "WikiText-2": "Salesforce/wikitext wikitext-2-raw-v1 (existing validation/test, first 50,000 tokens)"},
    "dataset_stats": DATA_INFO,
    "ppl_protocol": {"unit": "token stream of whole test split", "doc_joiner": repr(SEPARATOR_TEXT), "block_size": EVAL_BLOCK_SIZE,
                     "stride": "none (non-overlapping)", "incomplete_final_block": "dropped", "precision": "fp32", "batch_blocks": BATCH_BLOCKS,
                     "labels_used": False, "wikitext_protocol": "unchanged calculate_ppl, first 50,000 tokens, 512-token blocks"},
    "nonlinear_multiply_definition": "CGOS: per pair, lowest calibration hidden-state MSE among Remove/Average/Multiply (WikiText-2 calibration split)",
    "remove_definition": "fitted affine map of block L+1 alone (A2,b2), exactly as in the Assignment 2 notebook (not literal block deletion)",
    "average_definition": "0.5*(A1,b1)+0.5*(A2,b2), exactly as in the Assignment 2 notebook",
    "linear_multiply": "not tested",
    "team_numbering_assumed": TEAM_NUMBERING, "merge_plans": PLANS,
    "fit": {"fit_tokens": FIT_TOKENS, "calib_tokens": CALIB_TOKENS, "ridge": RIDGE, "fit_source": "WikiText-2 train (first 20k tokens); calibration = second half of train text"},
    "cgos_selection": CGOS_LOG,
    "seed": SEED, "device": str(DEVICE), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "versions": {"python": sys.version.split()[0], "torch": torch.__version__, "transformers": transformers.__version__, "datasets": datasets_pkg.__version__,
                 "numpy": np.__version__, "pandas": pd.__version__},
    "original_parameters": ORIG_PARAMS, "parameter_counts_by_steps": {int(s): int(param_by_steps[s]) for s in param_by_steps},
    "degradation_formula": "(Model PPL - Original PPL) / Original PPL * 100",
    "sanity_checks": checks, "git_commit": _git(),
}

def save(df, name, **kw): df.to_csv(os.path.join(OUT_DIR, name), index=False, **kw)
save(raw_df.drop(columns=["model_label"]), f"SGSC_Assignment2_Raw_Results{SUFFIX}.csv")
final_csv = raw_df[raw_df.steps > 0][["dataset", "method", "steps", "merge_plan_team_notation", "merge_plan_python_blocks", "ppl", "original_ppl",
                                      "degradation_pct", "parameter_count", "parameter_reduction_pct"]]
save(final_csv, f"SGSC_Assignment2_Final_Results{SUFFIX}.csv")
json.dump(metadata, open(os.path.join(OUT_DIR, f"SGSC_Assignment2_Evaluation_Metadata{SUFFIX}.json"), "w"), indent=2, default=str)
xlsx = os.path.join(OUT_DIR, f"SGSC_Assignment2_Final_Results{SUFFIX}.xlsx")
with pd.ExcelWriter(xlsx, engine="openpyxl") as w:
    table_a_fmt.to_excel(w, sheet_name="Table_A_PPL_Degradation", index=False)
    table_b.to_excel(w, sheet_name="Table_B_Proposed_Model", index=False)
    model_table.to_excel(w, sheet_name="Model_Table_All_Methods", index=False)
    raw_df.drop(columns=["model_label"]).to_excel(w, sheet_name="Raw_PPL_All", index=False)
    wiki_df.to_excel(w, sheet_name="WikiText2_Existing", index=False)
    cgos_df.to_excel(w, sheet_name="CGOS_Selection", index=False)
    pd.DataFrame([(k, json.dumps(v, default=str)) for k, v in metadata.items() if k != "sanity_checks"], columns=["key", "value"]).to_excel(w, sheet_name="Metadata", index=False)

pd.set_option("display.width", 200)
print("\n=== Original PPL ===");  print({d: round(orig["ppl"][d], 4) for d in ALL_EVALS})
print("\n=== Table A: PPL degradation (%) relative to original GPT-2 ===");  print(table_a_fmt.to_string(index=False))
print("\n=== Table B: proposed model (NonlinearMultiply = CGOS) ===");  print(table_b.to_string(index=False))
print("\nFiles in", OUT_DIR, ":", sorted(os.listdir(OUT_DIR)))

/tmp/ipykernel_1844/313222482.py:74: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.datetime.utcnow().isoformat() + "Z",



=== Original PPL ===
{'DBpedia': 31.3636, 'AG News': 63.3452, 'IMDb': 40.3047, 'WikiText-2 (validation)': 32.9046, 'WikiText-2 (test)': 38.4868}

=== Table A: PPL degradation (%) relative to original GPT-2 ===
Dataset  Steps (number of merged pairs) LinearMultiply (not tested)    Remove   Average NonlinearMultiply
DBpedia                               1                                 61.5%     43.2%             64.1%
DBpedia                               2                              97504.8%  32058.3%           1297.1%
DBpedia                               3                             322084.5%  50942.4%           1833.5%
DBpedia                               4                             475109.4% 110376.0%           2960.6%
DBpedia                               5                             696372.7% 240576.4%           6050.6%
AG News                               1                                112.9%     73.7%            127.2%
AG News                               2        

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [11]:
# optional: zip everything for Google Drive upload
import shutil
shutil.copy("README_Final_Evaluation.txt", OUT_DIR) if os.path.exists("README_Final_Evaluation.txt") else None
zip_path = shutil.make_archive("/content/SGSC_Assignment2_Final_Submission" + SUFFIX if os.path.isdir("/content") else "./SGSC_Assignment2_Final_Submission" + SUFFIX, "zip", OUT_DIR)
print("zip:", zip_path)
try:
    from google.colab import files; files.download(zip_path)
except Exception:
    pass

zip: /content/SGSC_Assignment2_Final_Submission.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>